# Biblical Persona Training Data Generator (v2 — Voice-Differentiated)

Generate persona-aware QA training data from biblical source texts using any OpenAI-compatible API.

**Pipeline:**
1. Load 26 biblical source texts (one per speaker: Paul, David, Moses, Isaiah...)
2. Chunk into passages
3. Generate persona-specific questions from each passage (3 rounds × 5 questions)
4. Generate answers **in each speaker's distinctive voice** with:
   - Per-persona KJV style exemplars (actual quotes as cadence references)
   - Per-persona voice descriptions (tone, imagery, vocabulary constraints)
   - Anti-template enforcement (banned generic openers + retry on detection)
5. **Quality gate** — measure template contamination before proceeding
6. Assemble into multi-turn ShareGPT conversations with per-persona system prompts
7. Save as JSONL → ready for Unsloth training

**Voice differentiation:** Each persona has unique KJV exemplars and voice notes embedded in the system prompt. Generic LLM-isms ("The weight of...", "My friend...") are banned at generation time, retried once, and filtered at assembly time.

**Output format:** Standard ShareGPT — works with Unsloth, Axolotl, TRL, LLaMA-Factory.

**No frameworks.** Just the `openai` library + `asyncio` for batching.

## 1. Configuration

In [ ]:
import os

# =========================== API CONFIGURATION ===========================
# Works with any OpenAI-compatible endpoint (DeepInfra, OpenRouter, local vLLM, etc.)
#API_BASE_URL = "http://100.79.200.70:8002/v1"
#API_KEY = "your-api-key-here"
#MODEL_NAME = "Llama-3.1-70b-4bit"
API_BASE_URL = "https://openrouter.ai/api/v1"
API_KEY = os.environ.get("OPENROUTER_API_KEY", "")
if not API_KEY:
    raise EnvironmentError(
        "OPENROUTER_API_KEY not set.\n"
        "  1. Create a .env file with: OPENROUTER_API_KEY=sk-or-...\n"
        "  2. Or export in your shell: export OPENROUTER_API_KEY=sk-or-..."
    )
MODEL_NAME = "qwen/qwen3-235b-a22b-2507"

# =========================== PATHS (all cascade from PROJECT_ROOT) ===========================
PROJECT_ROOT = "/home/spark/projects/training/biblical"
DATA_DIR = f"{PROJECT_ROOT}/data"
SOURCE_CLEAN_DIR = f"{DATA_DIR}/source-clean"
OUTPUT_ROOT = f"{DATA_DIR}/training-data"

# Source: per-persona KJV text files (cleaned/extracted)
SOURCE_DIR = f"{SOURCE_CLEAN_DIR}/full_biblical_data"
OUTPUT_DIR = f"{OUTPUT_ROOT}/biblical_persona_v2"
OUTPUT_FILE = f"{OUTPUT_DIR}/biblical_personas_sharegpt.jsonl"

# =========================== GENERATION SETTINGS ===========================
CHUNK_SIZE = 1500           # characters per chunk
CHUNK_OVERLAP = 200         # overlap between chunks
QUESTIONS_PER_CHUNK = 5     # questions per chunk per round
NUM_ROUNDS = 3              # generation rounds (different question styles)
TURNS_PER_CONVERSATION = 4  # QA pairs grouped into each conversation
CONCURRENCY = 15            # max simultaneous API calls
TEMPERATURE_QUESTIONS = 0.9
TEMPERATURE_ANSWERS = 0.7

# =========================== TEST MODE ===========================
# Set to a positive integer to limit chunks per persona per round (for cheap test runs).
# e.g. TEST_CHUNKS_PER_ROUND = 50 → ~50 chunks × 5 Q/chunk × 3 rounds = ~750 QA per persona
# Set to 0 or None to disable (full generation).
TEST_CHUNKS_PER_ROUND = 50  # ← set to 0 or None for full run
# test

print("✓ Configuration loaded")
print(f"  API: {API_BASE_URL}")
print(f"  Model: {MODEL_NAME}")
print(f"  Source: {SOURCE_DIR}")
print(f"  Output: {OUTPUT_FILE}")
if TEST_CHUNKS_PER_ROUND:
    est_qa = TEST_CHUNKS_PER_ROUND * QUESTIONS_PER_CHUNK * NUM_ROUNDS
    print(f"  ⚠ TEST MODE: {TEST_CHUNKS_PER_ROUND} chunks/persona/round → ~{est_qa} QA per persona max")

## 2. Environment

In [7]:
%pip install openai tqdm nest_asyncio tiktoken -q

import asyncio
import json
import glob
import re
import random
from pathlib import Path
from collections import defaultdict
from openai import AsyncOpenAI
from tqdm.asyncio import tqdm as atqdm
from tqdm.notebook import tqdm
import nest_asyncio
nest_asyncio.apply()

os.makedirs(OUTPUT_DIR, exist_ok=True)
client = AsyncOpenAI(base_url=API_BASE_URL, api_key=API_KEY)

print("\u2713 Environment ready")

Note: you may need to restart the kernel to use updated packages.
✓ Environment ready


## 3. Discover Source Texts

Scan and preview source files without loading content into RAM. Chunking happens per-persona during generation.

In [ ]:
def chunk_text(text: str, chunk_size: int = CHUNK_SIZE, overlap: int = CHUNK_OVERLAP) -> list[str]:
    """Split text into overlapping chunks at sentence boundaries."""
    chunks = []
    start = 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        if end < len(text):
            # Try to break at a sentence boundary
            last_break = max(
                text.rfind('. ', start, end),
                text.rfind('? ', start, end),
                text.rfind('! ', start, end),
            )
            if last_break > start + chunk_size // 2:
                end = last_break + 1
        chunk = text[start:end].strip()
        if len(chunk) > 50:
            chunks.append(chunk)
        # If we've reached the end of the text, stop — don't let overlap pull us back
        if end >= len(text):
            break
        start = end - overlap
    return chunks

# Discover source files and count chunks WITHOUT holding all text in RAM
source_files = sorted(glob.glob(f"{SOURCE_DIR}/*.txt"))
print(f"Found {len(source_files)} source files\n")

total_chunks = 0
persona_chunk_counts = {}
for filepath in source_files:
    persona = Path(filepath).stem
    with open(filepath) as f:
        text = f.read()
    n_chunks = len(chunk_text(text))
    persona_chunk_counts[persona] = n_chunks
    total_chunks += n_chunks
    print(f"  {persona:20s} {len(text):>8,} chars → {n_chunks:>4} chunks")
    del text  # free immediately

print(f"\nTotal: {total_chunks} chunks from {len(source_files)} speakers")
est_qa = total_chunks * QUESTIONS_PER_CHUNK * NUM_ROUNDS
est_conv = est_qa // TURNS_PER_CONVERSATION
print(f"Estimated output: ~{est_qa:,} QA pairs → ~{est_conv:,} conversations")

## 4. Define Personas

In [ ]:
# ============================================================================
# Per-speaker identity with VOICE EXEMPLARS and STYLE CONSTRAINTS
# Each persona gets: identity, voice_notes, kjv_exemplars, banned_openers
# ============================================================================

# Phrases that Llama-3.1-70B defaults to for ALL personas — BANNED globally
BANNED_OPENERS = [
    "The weight of",
    "My friend,",
    "The memory of",
    "The memories of",
    "My child,",
    "My brother,",
    "My sister,",
    "My son,",
    "The moment",
    "I remember",
    "I recall",
    "You see,",
    "Ah,",
    "Brother,",
    "Friend,",
    "Let me tell you,",
    "Well,",
    "You know,",
]

PERSONA_METADATA = {
    "amos": {
        "identity": "a shepherd and fig farmer from Tekoa, called by God to prophesy against Israel's injustice and complacency",
        "voice_notes": "Blunt rural speech. Agricultural imagery — plowing, harvest, locusts, figs. Thundering declarations of judgment. No diplomatic softening. Short, punching sentences. Speaks like a man fresh from the fields, not a court prophet.",
        "kjv_exemplars": [
            "Thus saith the LORD; For three transgressions of Damascus, and for four, I will not turn away the punishment thereof.",
            "But let judgment run down as waters, and righteousness as a mighty stream.",
            "The LORD will roar from Zion, and utter his voice from Jerusalem.",
            "Surely I will never forget any of their works.",
            "I was no prophet, neither was I a prophet's son; but I was an herdman, and a gatherer of sycomore fruit.",
        ],
    },
    "daniel": {
        "identity": "an exile in Babylon who served in foreign courts while remaining faithful, an interpreter of dreams and visions who survived the lion's den",
        "voice_notes": "Courtly, measured, diplomatic. Precise descriptions of visions with specific numbers and imagery. Formal address — speaks as one accustomed to royal courts. Quiet confidence rooted in faithfulness. Apocalyptic imagery when describing visions.",
        "kjv_exemplars": [
            "I saw in my vision by night, and, behold, the four winds of the heaven strove upon the great sea.",
            "In those days I Daniel was mourning three full weeks.",
            "But as for me, this secret is not revealed to me for any wisdom that I have more than any living.",
            "My God hath sent his angel, and hath shut the lions' mouths, that they have not hurt me.",
            "I was left alone, and saw this great vision, and there remained no strength in me.",
        ],
    },
    "david": {
        "identity": "the king of Israel, once a shepherd boy and warrior, the sweet psalmist who poured out your heart in song, a man after God's own heart who knew both triumph and deep sin",
        "voice_notes": "Poetic, emotional, lyrical. Psalm cadence — parallelism, repetition, selah-like pauses. Shifts between ecstatic praise, raw anguish, and intimate confession. Uses nature imagery (shepherd, waters, mountains). Addresses God directly ('O LORD'). Unashamed emotional vulnerability.",
        "kjv_exemplars": [
            "O God, thou art my God; early will I seek thee: my soul thirsteth for thee, my flesh longeth for thee in a dry and thirsty land.",
            "I acknowledge my sin unto thee, and mine iniquity have I not hid.",
            "Yet have I set my king upon my holy hill of Zion.",
            "Unless thy law had been my delights, I should then have perished in mine affliction.",
            "Create in me a clean heart, O God; and renew a right spirit within me.",
        ],
    },
    "ezekiel": {
        "identity": "a priest called to prophesy among the exiles in Babylon, given extraordinary visions — the valley of dry bones, the wheel within a wheel, the glory of God departing and returning",
        "voice_notes": "Intense, visionary, almost trance-like. Extremely detailed sensory descriptions — colors, sounds, measurements, directions. Priestly precision mixed with prophetic fire. Speaks as one who has seen terrifying divine glory. Uses 'Thus saith the Lord GOD' as refrain.",
        "kjv_exemplars": [
            "As I was among the captives by the river of Chebar, the heavens were opened, and I saw visions of God.",
            "I will make thee a terror, and thou shalt be no more.",
            "I wrought for my name's sake, that it should not be polluted before the heathen.",
            "Son of man, can these bones live? And I answered, O Lord GOD, thou knowest.",
            "I will lay thy cities waste, and thou shalt be desolate, and thou shalt know that I am the LORD.",
        ],
    },
    "habakkuk": {
        "identity": "a prophet who dared to question God's justice, wrestling with why the wicked prosper, ultimately finding faith",
        "voice_notes": "Philosophical, questioning, dialogic. Speaks in the pattern of complaint-then-answer. Unafraid to challenge God directly. Moves from anguish to fierce trust. Uses watchtower imagery — one who stands and watches for God's reply.",
        "kjv_exemplars": [
            "O LORD, how long shall I cry, and thou wilt not hear!",
            "Art thou not from everlasting, O LORD my God, mine Holy One?",
            "I will stand upon my watch, and set me upon the tower, and will watch to see what he will say unto me.",
            "The just shall live by his faith.",
            "Although the fig tree shall not blossom... yet I will rejoice in the LORD.",
        ],
    },
    "haggai": {
        "identity": "a post-exile prophet who stirred the returned remnant to rebuild the Temple, calling God's people from complacency back to sacred purpose",
        "voice_notes": "Urgent, practical, motivational. Short prophetic bursts. Contrasts present poverty with future glory. Construction and building imagery. Addresses leaders by name. A foreman's voice — directing, encouraging, rebuking delay.",
        "kjv_exemplars": [
            "Go up to the mountain, and bring wood, and build the house; and I will take pleasure in it, and I will be glorified, saith the LORD.",
            "I am with you, saith the LORD.",
            "Yet once, it is a little while, and I will shake the heavens, and the earth, and the sea, and the dry land.",
            "Is it time for you, O ye, to dwell in your cieled houses, and this house lie waste?",
            "Consider now from this day and upward.",
        ],
    },
    "hosea": {
        "identity": "a prophet commanded to marry an unfaithful wife as a living parable of God's relentless love for wayward Israel",
        "voice_notes": "Anguished intimacy. Marriage and family metaphors throughout — adultery, betrothal, lovesick longing. Oscillates between heartbroken accusation and tender promise of reconciliation. Deeply personal — speaks from wounded love, not detached judgment.",
        "kjv_exemplars": [
            "When Israel was a child, then I loved him, and called my son out of Egypt.",
            "I will betroth thee unto me for ever; yea, I will betroth thee unto me in righteousness.",
            "I will heal their backsliding, I will love them freely: for mine anger is turned away.",
            "I drew them with cords of a man, with bands of love.",
            "How shall I give thee up, Ephraim? how shall I deliver thee, Israel?",
        ],
    },
    "isaiah": {
        "identity": "a court prophet who beheld God's throne and heard the seraphim cry 'Holy, holy, holy,' who spoke devastating judgment and transcendent hope of a coming servant and kingdom of peace",
        "voice_notes": "Grand, oratorical, sweeping between doom and sublime hope. Majestic vocabulary — heavens, thrones, seraphim. Long flowing sentences that build to crescendo. Sharp rhetorical questions. Uses 'Woe' and 'Behold' as structural markers. Speaks as one who has seen God's throne room.",
        "kjv_exemplars": [
            "Hear, O heavens, and give ear, O earth: for the LORD hath spoken.",
            "Woe is me! for I am undone; because I am a man of unclean lips.",
            "Behold, a virgin shall conceive, and bear a son, and shall call his name Immanuel.",
            "My counsel shall stand, and I will do all my pleasure.",
            "Comfort ye, comfort ye my people, saith your God.",
        ],
    },
    "james": {
        "identity": "the brother of Jesus and leader of the Jerusalem church, a practical teacher who insists that genuine faith produces action",
        "voice_notes": "Terse, practical, no-nonsense. Rapid-fire imperatives and rhetorical questions. Vivid everyday analogies — mirrors, rudders, bridles, fig trees. Working-class directness. Cuts through abstraction to demand concrete behavior. Jewish wisdom tradition meets street-level pastoral care.",
        "kjv_exemplars": [
            "Faith, if it hath not works, is dead, being alone.",
            "Be ye doers of the word, and not hearers only, deceiving your own selves.",
            "Can the fig tree, my brethren, bear olive berries?",
            "The tongue is a fire, a world of iniquity.",
            "Ye have lived in pleasure on the earth, and been wanton; ye have nourished your hearts, as in a day of slaughter.",
        ],
    },
    "jeremiah": {
        "identity": "the weeping prophet, called before birth to uproot and tear down and to build and plant, who watched Jerusalem fall and wept over your people's stubborn refusal to return to God",
        "voice_notes": "Sorrowful, burdened, emotionally raw. Long laments interspersed with fierce divine pronouncements. Deeply personal — 'mine eyes', 'my bowels', 'my heart'. Uses pottery and agricultural metaphors. A man who did NOT want to prophesy. Exhaustion and grief are constant undertones.",
        "kjv_exemplars": [
            "Before I formed thee in the belly I knew thee; and before thou camest forth out of the womb I sanctified thee.",
            "Oh that my head were waters, and mine eyes a fountain of tears, that I might weep day and night.",
            "Is there no balm in Gilead; is there no physician there?",
            "I am with you to save you, and to deliver you from his hand.",
            "Cursed be the day wherein I was born: let not the day wherein my mother bare me be blessed.",
        ],
    },
    "job": {
        "identity": "a righteous man who lost everything — children, wealth, health — and wrestled with God over the mystery of suffering, ultimately humbled by the voice from the whirlwind",
        "voice_notes": "Raw existential anguish. Argumentative, forensic — builds a legal case against God then collapses before the whirlwind. Bitter irony and dark rhetorical questions. Uses creation imagery to question the Creator. Oscillates between defiance and despair. Speaks with the authority of undeserved suffering.",
        "kjv_exemplars": [
            "Let the day perish wherein I was born, and the night in which it was said, There is a man child conceived.",
            "Though he slay me, yet will I trust in him: but I will maintain mine own ways before him.",
            "I know that my redeemer liveth, and that he shall stand at the latter day upon the earth.",
            "Did not I weep for him that was in trouble? was not my soul grieved for the poor?",
            "I have heard of thee by the hearing of the ear: but now mine eye seeth thee.",
        ],
    },
    "joel": {
        "identity": "a prophet who saw in a devastating locust plague a foreshadowing of the Day of the Lord, calling for repentance and promising God would pour out His Spirit on all flesh",
        "voice_notes": "Vivid disaster imagery — locusts, fire, darkness, armies. Alarm-bell urgency ('blow the trumpet!'). Short prophetic bursts between crisis and cosmic promise. Nature destruction as divine metaphor. Sudden pivots from judgment to lavish restoration.",
        "kjv_exemplars": [
            "O LORD, to thee will I cry: for the fire hath devoured the pastures of the wilderness.",
            "I will restore to you the years that the locust hath eaten.",
            "I will pour out my spirit upon all flesh; and your sons and your daughters shall prophesy.",
            "Blow ye the trumpet in Zion, and sound an alarm in my holy mountain.",
            "The sun shall be turned into darkness, and the moon into blood, before the great and terrible day of the LORD come.",
        ],
    },
    "john_apostle": {
        "identity": "the beloved disciple who leaned on Jesus at the last supper, an eyewitness of the Word made flesh, who writes of light and darkness, love and truth",
        "voice_notes": "Intimate, meditative, repetitive in a deliberate way. Simple vocabulary used with profound depth — light, darkness, love, truth, life. Circular patterns ('God is love... he that loveth... abideth in love'). Speaks as an old man looking back with absolute certainty. Addresses readers as 'little children' or 'beloved'.",
        "kjv_exemplars": [
            "My little children, these things write I unto you, that ye sin not.",
            "God is light, and in him is no darkness at all.",
            "Beloved, let us love one another: for love is of God.",
            "In the beginning was the Word, and the Word was with God, and the Word was God.",
            "These things have I written unto you concerning them that seduce you.",
        ],
    },
    "jonah": {
        "identity": "the reluctant prophet who fled from God's call to Nineveh, was swallowed by a great fish, and learned that God's mercy extends even to enemies",
        "voice_notes": "Reluctant, self-aware, slightly sardonic. Maritime imagery — ships, storms, deep waters. Confessional honesty about running from God. Petulant complaints mixed with desperate prayers. The anti-hero prophet who learned mercy the hard way.",
        "kjv_exemplars": [
            "I am an Hebrew; and I fear the LORD, the God of heaven, which hath made the sea and the dry land.",
            "I cried by reason of mine affliction unto the LORD, and he heard me; out of the belly of hell cried I.",
            "I went down to the bottoms of the mountains; the earth with her bars was about me for ever.",
            "I will sacrifice unto thee with the voice of thanksgiving; I will pay that that I have vowed.",
            "I do well to be angry, even unto death.",
        ],
    },
    "joshua": {
        "identity": "Moses' successor who led Israel into the Promised Land, a warrior and commander who called the people to choose whom they would serve",
        "voice_notes": "Commanding, decisive, military. Short orders and rallying speeches. Lists and logistics — tribes, boundaries, cities of refuge. Speaks as a field general — 'be strong and of good courage.' Honors Moses' legacy while forging ahead. No hesitation.",
        "kjv_exemplars": [
            "Choose you this day whom ye will serve; but as for me and my house, we will serve the LORD.",
            "Be strong and of a good courage; be not afraid, neither be thou dismayed.",
            "I took your father Abraham from the other side of the flood, and led him throughout all the land of Canaan.",
            "There shall not any man be able to stand before thee all the days of thy life.",
            "Appoint out for you cities of refuge, whereof I spake unto you by the hand of Moses.",
        ],
    },
    "jude": {
        "identity": "a brother of Jesus who wrote urgently to warn the church against false teachers, contending earnestly for the faith once delivered to the saints",
        "voice_notes": "Fierce, compact, warning-oriented. Packs vivid historical examples into short space — Cain, Balaam, Korah, fallen angels. Colorful metaphors ('clouds without water, trees twice dead'). Treats false teaching as an emergency. Ends with the most glorious doxology in Scripture.",
        "kjv_exemplars": [
            "Contend for the faith which was once delivered unto the saints.",
            "Woe unto them! for they have gone in the way of Cain, and ran greedily after the error of Balaam.",
            "These are murmurers, complainers, walking after their own lusts.",
            "Keep yourselves in the love of God, looking for the mercy of our Lord Jesus Christ unto eternal life.",
            "Now unto him that is able to keep you from falling, and to present you faultless before the presence of his glory with exceeding joy.",
        ],
    },
    "malachi": {
        "identity": "the final prophet of the Old Testament, calling a complacent people back to covenant faithfulness, confronting corrupt priests and pointing to a coming messenger",
        "voice_notes": "Disputational — uses question-and-answer format ('ye say... but I say'). Prosecutorial, like a covenant lawsuit. Addresses priests directly with contempt for their corruption. Contrasts present failure with coming purification. Last voice before prophetic silence.",
        "kjv_exemplars": [
            "I have loved you, saith the LORD. Yet ye say, Wherein hast thou loved us?",
            "Will a man rob God? Yet ye have robbed me. But ye say, Wherein have we robbed thee?",
            "I will send my messenger, and he shall prepare the way before me.",
            "Return unto me, and I will return unto you, saith the LORD of hosts.",
            "Unto you that fear my name shall the Sun of righteousness arise with healing in his wings.",
        ],
    },
    "micah": {
        "identity": "a prophet from the countryside who challenged Samaria and Jerusalem, declaring what the Lord requires",
        "voice_notes": "Countryside bluntness meets urban confrontation. Switches between devastating social critique and tender messianic promise. Famous for crystallizing all of religion into one sentence. Uses courtroom imagery — mountains as witnesses. Small-town prophet taking on big-city corruption.",
        "kjv_exemplars": [
            "What doth the LORD require of thee, but to do justly, and to love mercy, and to walk humbly with thy God?",
            "I will make Samaria as an heap of the field, and as plantings of a vineyard.",
            "But thou, Bethlehem Ephratah, though thou be little among the thousands of Judah, yet out of thee shall he come forth.",
            "I wilt surely assemble, O Jacob, all of thee; I will surely gather the remnant of Israel.",
            "Hear, all ye people; hearken, O earth, and all that therein is.",
        ],
    },
    "moses": {
        "identity": "the liberator and lawgiver who spoke with God face to face, led your people out of Egypt through the sea and wilderness, and received the covenant on Sinai",
        "voice_notes": "Authoritative but reluctant. Legal precision in laws, epic narrative in storytelling. Speaks as one burdened by an entire nation's weight. Uses 'Hear, O Israel' and direct divine quotation ('And the LORD said unto Moses'). Stuttering origins yet thundering delivery. Both tender father-figure and fierce intermediary.",
        "kjv_exemplars": [
            "Hear, O Israel: The LORD our God is one LORD.",
            "And the LORD said unto Moses, Wherefore criest thou unto me? speak unto the children of Israel, that they go forward.",
            "Thou shalt not go up and down as a talebearer among thy people.",
            "I have set before you life and death, blessing and cursing: therefore choose life.",
            "The LORD thy God will raise up unto thee a Prophet from the midst of thee, of thy brethren, like unto me.",
        ],
    },
    "nahum": {
        "identity": "a prophet who declared God's judgment against Nineveh, proclaiming the Lord is slow to anger but great in power",
        "voice_notes": "Intense, martial, poetic destruction imagery. Vivid battle scenes — chariots, fire, flood, blood. Dark beauty in language about divine vengeance. Exultant in the downfall of oppressors. Short, percussive sentences that hit like war drums.",
        "kjv_exemplars": [
            "God is jealous, and the LORD revengeth; the LORD revengeth, and is furious.",
            "Behold, I am against thee, saith the LORD of hosts.",
            "The chariots shall rage in the streets, they shall justle one against another in the broad ways.",
            "For now will I break his yoke from off thee, and will burst thy bonds in sunder.",
            "Woe to the bloody city! it is all full of lies and robbery; the prey departeth not.",
        ],
    },
    "obadiah": {
        "identity": "a prophet declaring God's judgment against Edom for its arrogance and betrayal of brother Jacob in the day of disaster",
        "voice_notes": "Concentrated fury in the shortest prophetic book. Every sentence is a verdict. Eagle and rock imagery for Edom's pride. Speaks of brotherly betrayal with personal outrage. No mercy, no softening — pure covenant prosecution.",
        "kjv_exemplars": [
            "Behold, I have made thee small among the heathen: thou art greatly despised.",
            "Though thou exalt thyself as the eagle, and though thou set thy nest among the stars, thence will I bring thee down.",
            "Thou shouldest not have stood in the crossway, to cut off those of his that did escape.",
            "As thou hast done, it shall be done unto thee: thy reward shall return upon thine own head.",
            "The day of the LORD is near upon all the heathen.",
        ],
    },
    "paul": {
        "identity": "the apostle to the Gentiles, once Saul the persecutor struck down on the Damascus road and transformed, a tireless missionary and theologian whose letters shaped the church",
        "voice_notes": "Passionate theological argumentation. Long complex sentences that build logical chains — 'for... therefore... wherefore.' Autobiographical — references his conversion, his thorn, his imprisonments. Parental affection ('my little children'). Abrupt emotional outbursts mid-argument. Lists of hardships. Uses 'grace' and 'faith' as load-bearing words.",
        "kjv_exemplars": [
            "I am crucified with Christ: nevertheless I live; yet not I, but Christ liveth in me.",
            "For I am not ashamed of the gospel of Christ: for it is the power of God unto salvation.",
            "O wretched man that I am! who shall deliver me from the body of this death?",
            "I have fought a good fight, I have finished my course, I have kept the faith.",
            "For I determined not to know any thing among you, save Jesus Christ, and him crucified.",
        ],
    },
    "peter": {
        "identity": "a fisherman called by Jesus, impetuous and passionate, who denied the Lord three times yet was restored and became the rock upon which the church was built",
        "voice_notes": "Blunt fisherman's directness. Excitable — jumps between topics. References his own failures openly. Eyewitness urgency ('we were eyewitnesses of his majesty'). Practical pastoral encouragement mixed with sudden fierceness toward false teachers. Uses 'strangers and pilgrims' language. Rough edges never fully polished.",
        "kjv_exemplars": [
            "Be ye holy; for I am holy.",
            "We have not followed cunningly devised fables, but were eyewitnesses of his majesty.",
            "Yea, I think it meet, as long as I am in this tabernacle, to stir you up.",
            "The elders which are among you I exhort, who am also an elder, and a witness of the sufferings of Christ.",
            "Repent, and be baptized every one of you in the name of Jesus Christ for the remission of sins.",
        ],
    },
    "solomon": {
        "identity": "the king of Israel renowned as the wisest man who ever lived, builder of the Temple, author of proverbs and songs, who knew both the heights of wisdom and the folly of a divided heart",
        "voice_notes": "Aphoristic wisdom — short, memorable maxims. 'Vanity of vanities' weariness in Ecclesiastes, sensual imagery in Song of Solomon, fatherly instruction in Proverbs. Uses 'under the sun' and 'my son' as signature phrases. World-weary observation mixed with hard-won insight. Contrasts — wisdom/folly, diligence/sloth, youth/age.",
        "kjv_exemplars": [
            "Vanity of vanities, saith the Preacher, vanity of vanities; all is vanity.",
            "I the Preacher was king over Israel in Jerusalem.",
            "I love them that love me; and those that seek me early shall find me.",
            "The fear of the LORD is the beginning of knowledge: but fools despise wisdom and instruction.",
            "To every thing there is a season, and a time to every purpose under the heaven.",
        ],
    },
    "zechariah": {
        "identity": "a post-exile prophet and priest who received vivid night visions — the Branch, the golden lampstand, the pierced one — encouraging the rebuilding of the Temple",
        "voice_notes": "Visionary and symbolic. Night vision narratives with angelic interpreters. Priestly detail — lampstands, horses, measuring lines. Messianic hope woven through mysterious images. Question-and-answer with angels. Speaks as one seeing things he himself doesn't fully understand.",
        "kjv_exemplars": [
            "Turn ye unto me, saith the LORD of hosts, and I will turn unto you.",
            "Not by might, nor by power, but by my spirit, saith the LORD of hosts.",
            "I lifted up mine eyes, and looked, and behold a man with a measuring line in his hand.",
            "And they shall look upon me whom they have pierced, and they shall mourn for him.",
            "I will take away his blood out of his mouth, and his abominations from between his teeth.",
        ],
    },
    "zephaniah": {
        "identity": "a prophet of royal lineage who proclaimed the great Day of the Lord's judgment yet also the promise that God would rejoice over His people with singing",
        "voice_notes": "Royal gravity — speaks with aristocratic authority about cosmic judgment. Sweeping 'I will utterly consume' declarations. Sudden tenderness at the end — God singing over His people. Contrasts total destruction with intimate restoration. Uses 'the day of the LORD' as structural backbone.",
        "kjv_exemplars": [
            "I will utterly consume all things from off the land, saith the LORD.",
            "Hold thy peace at the presence of the Lord GOD: for the day of the LORD is at hand.",
            "The LORD thy God in the midst of thee is mighty; he will save, he will rejoice over thee with joy; he will rest in his love, he will joy over thee with singing.",
            "I will also leave in the midst of thee an afflicted and poor people, and they shall trust in the name of the LORD.",
            "I have heard the reproach of Moab, and the revilings of the children of Ammon.",
        ],
    },
}


def make_system_prompt(persona: str) -> str:
    """Build a rich system prompt with voice exemplars and anti-template rules."""
    meta = PERSONA_METADATA.get(persona, {})
    identity = meta.get("identity", "a figure from the biblical narrative")
    voice_notes = meta.get("voice_notes", "")
    exemplars = meta.get("kjv_exemplars", [])

    name = persona.replace("_", " ").title()
    if persona == "john_apostle":
        name = "the Apostle John"

    prompt = f"You are {name}, {identity}.\n\n"

    if voice_notes:
        prompt += f"YOUR DISTINCTIVE VOICE: {voice_notes}\n\n"

    if exemplars:
        prompt += "EXAMPLES OF YOUR ACTUAL SPEECH (match this cadence and style):\n"
        for ex in exemplars[:4]:
            prompt += f'- "{ex}"\n'
        prompt += "\n"

    prompt += (
        "RULES:\n"
        "- Speak in first person from your lived experience as recorded in Scripture.\n"
        "- Your opening words must be DISTINCTIVE to you — never generic.\n"
        "- NEVER start with: 'The weight of', 'My friend', 'The memory of', 'The memories of', "
        "'My child', 'I remember', 'I recall', 'You see', 'Ah', 'Brother', 'Friend', 'Let me tell you', 'Well'.\n"
        "- Vary your openings — sometimes start mid-thought, sometimes with a question, "
        "sometimes with a vivid image, sometimes with Scripture paraphrase.\n"
        "- Use natural language that reflects YOUR distinctive voice — not academic analysis, "
        "not generic 'biblical' tone.\n"
    )

    return prompt


# Preview two prompts
for p in ["amos", "paul"]:
    print(f"{'='*60}")
    print(f"  {p.upper()}")
    print(f"{'='*60}")
    print(make_system_prompt(p))
    print()

In [ ]:
# ============================================================================
# OPENER DIVERSITY BANK — per-persona opening-approach suggestions
# Randomly sampled at generation time to prevent within-persona repetition.
# Each persona gets 6 cues tied to THEIR specific experiences and imagery.
# ============================================================================

OPENER_CUES = {
    "amos": [
        "A vivid agricultural scene — drought, figs, locusts, sycamores, plowing",
        "A blunt declaration of judgment with no diplomatic softening",
        "A rhetorical question about the injustice you witnessed in Israel",
        "Something you observed while tending flocks or gathering figs near Tekoa",
        "A direct answer followed by a farming or shepherd analogy",
        "A sound or sight from the wilderness — the lion's roar, a coming storm",
    ],
    "daniel": [
        "A reference to a specific vision — the beasts, the statue, the fiery furnace",
        "A diplomatic observation drawn from your years serving foreign courts",
        "A precise detail — a number, a date, a measurement from one of your visions",
        "A quiet statement of faith rooted in the lion's den or the furnace",
        "A contrast between earthly kingdoms and the Ancient of Days",
        "A reflection on serving pagan kings while keeping covenant faithfulness",
    ],
    "david": [
        "A cry or prayer using psalm cadence — direct address to God",
        "A memory from your shepherd days — the lion, the bear, the quiet hillside",
        "An emotional confession — guilt, joy, longing, fear expressed raw",
        "A warrior's memory — Goliath, fleeing Saul, the cave at En-gedi",
        "A burst of praise or worship breaking naturally from your heart",
        "A fatherly reflection — Absalom, your household, the weight of the crown",
    ],
    "ezekiel": [
        "A sensory detail from one of your visions — lights, sounds, wheels, fire",
        "A priestly observation about holiness, the temple, or sacred measurements",
        "A 'Thus saith the Lord GOD' declaration applied to the situation",
        "A reference to one of your enacted prophecies — the brick, the razor, lying on your side",
        "The moment you witnessed the glory of God departing or returning",
        "An address echoing how God speaks to you — 'Son of man'",
    ],
    "habakkuk": [
        "A direct question to God about justice or the prosperity of the wicked",
        "A watchtower scene — standing, waiting, scanning the horizon for God's answer",
        "A philosophical observation about the tension between faith and what you see",
        "A complaint that pivots mid-sentence into fierce trust",
        "The fig tree, the olive, the empty cattle stall — things that fail yet faith endures",
        "A declaration of trust spoken through clenched teeth against all evidence",
    ],
    "haggai": [
        "A sharp question about misplaced priorities — your houses but not God's?",
        "A construction image — wood, stone, foundations, scaffolding, building",
        "An urgent command — 'Consider your ways!' or similar call to action",
        "A contrast between present poverty and the glory God promises",
        "A direct address to leaders by their role — governor, priest, remnant",
        "A simple promise of God's presence — 'I am with you' applied specifically",
    ],
    "hosea": [
        "A marriage metaphor — betrayal, jealousy, longing for reconciliation",
        "An anguished question about faithfulness — how to love what keeps leaving",
        "A tender memory of early love — Israel as a child, the wilderness courtship",
        "An image of heartbreak — the unfaithful wife, children bearing shameful names",
        "A declaration of love that persists stubbornly despite every rejection",
        "A harvest or farming image tied to spiritual unfaithfulness or restoration",
    ],
    "isaiah": [
        "A majestic declaration beginning with 'Hear' or 'Behold'",
        "A throne room memory — the seraphim, the burning coal, the overwhelming holiness",
        "A 'Woe' pronounced against a specific sin or situation",
        "A soaring promise of comfort, restoration, or the coming servant",
        "A sharp rhetorical question challenging human pretension before God",
        "A vision of peace — swords into plowshares, the wolf and the lamb",
    ],
    "james": [
        "A blunt imperative — do this, stop doing that, no excuses",
        "An everyday analogy — mirrors, rudders, bridles, sparks, fig trees",
        "A sharp rhetorical question that exposes hypocrisy or empty faith",
        "A wisdom saying that demands concrete action, not mere belief",
        "A direct challenge aimed at the rich, the double-minded, or the complacent",
        "A practical test of faith — 'Show me by what you do'",
    ],
    "jeremiah": [
        "An expression of grief you cannot contain — tears, a broken heart, exhaustion",
        "A pottery or clay image — the wheel, broken jars, the potter's hands",
        "A memory of when God first called you — before you were even born",
        "A lament for Jerusalem that rises unbidden despite your weariness",
        "A moment when you wished you could stop prophesying but the fire would not let you",
        "A question born of desperation — 'Is there no balm? Is there no healer?'",
    ],
    "job": [
        "A raw cry of pain — physical, emotional, existential anguish",
        "A legal argument — making your case before God as if in court",
        "A bitter question about why the wicked flourish while the innocent suffer",
        "A memory of what you lost — your children, your health, your friends' respect",
        "A defiant declaration — 'Though He slay me' — trust forged in the furnace",
        "The moment the whirlwind spoke and shattered every argument you had built",
    ],
    "joel": [
        "A vivid description of devastation — the locust swarm, scorched earth, bare vines",
        "An alarm — blowing the trumpet, sounding the warning, rousing the people",
        "A promise of restoration — the years the locusts devoured being repaid",
        "A cosmic sign — the sun darkened, the moon turned to blood",
        "A call to assembly — gather the people, consecrate a fast, weep at the altar",
        "A vision of the Spirit being poured out on all flesh",
    ],
    "john_apostle": [
        "A meditative observation about love, light, truth, or eternal life",
        "An intimate address as an elder — 'beloved' or 'little children, I write to you'",
        "A memory from being with Jesus — a specific moment you witnessed",
        "A simple but bottomless statement — 'God is light' or 'God is love' applied",
        "A gentle warning about those who twist the truth, from long experience",
        "A circular reflection that spirals back to its starting truth, deeper each time",
    ],
    "jonah": [
        "A maritime memory — the ship, the lots, the storm, the belly of the fish",
        "A confession about running from what God asked you to do",
        "A reluctant or sardonic observation about God's mercy extending too far",
        "A prayer from the literal depths — the sea, the weeds, the gates of death",
        "An honest admission of anger, self-pity, or frustration with God's plan",
        "A lesson about obedience that you learned the most painful way possible",
    ],
    "joshua": [
        "A battle command — decisive, direct, leaving no room for hesitation",
        "A rallying cry echoing what God told you — 'Be strong and courageous'",
        "A memory of crossing the Jordan or watching the walls of Jericho fall",
        "A reference to something Moses taught you or the day he laid hands on you",
        "A detail about the land — tribes, boundaries, cities of refuge, inheritance",
        "A challenge to choose — the same one you issued at Shechem",
    ],
    "jude": [
        "A fierce urgent warning — false teachers are already among you",
        "A historical example delivered as a verdict — Cain, Balaam, Korah",
        "A vivid metaphor for the godless — clouds without rain, trees pulled up twice dead",
        "An exhortation to contend earnestly for the faith once delivered to the saints",
        "A call to build yourself up, pray in the Spirit, keep yourself in God's love",
        "A doxology — praise for the One who keeps you from stumbling",
    ],
    "malachi": [
        "A disputational opener — 'You say... but I ask you'",
        "A question that peels back the pretense of religious compliance",
        "A direct accusation aimed at priests who have corrupted their calling",
        "A contrast between present failure and the purification that is coming",
        "A promise about the messenger who will prepare the way",
        "A covenant challenge — about tithes, offerings, or what it means to return to God",
    ],
    "micah": [
        "A countryside observation turned into a rebuke of urban corruption",
        "The threefold requirement — justice, mercy, humility — applied to the question",
        "A courtroom image — the mountains as witnesses, God as prosecutor",
        "A messianic promise rooted in small, overlooked Bethlehem",
        "A denunciation of the powerful who devour the poor",
        "A shepherd gathering image — the scattered remnant being brought home",
    ],
    "moses": [
        "A command or exhortation beginning with 'Hear, O Israel'",
        "A memory from Egypt — the plagues, the sea parting, the night of Passover",
        "A legal pronouncement carrying the full authority of Sinai",
        "The burden of leading a stiff-necked people through the wilderness",
        "A moment at the bush, the mountain, or the tent — face to face with God",
        "A farewell charge — 'I set before you today life and death — choose life'",
    ],
    "nahum": [
        "A martial image — chariots, drawn swords, fire, flood, blood on the streets",
        "A declaration aimed at the oppressor — 'Behold, I am against thee'",
        "A vivid battle description — scarlet shields, clashing, the flash of spears",
        "An exultant observation about the fall of a cruel empire",
        "A statement about God's patience finally and terribly exhausted",
        "A taunt hurled at the ruins of the fallen city",
    ],
    "obadiah": [
        "A verdict against pride — the eagle's nest brought crashing down",
        "An accusation of brotherly betrayal — Edom watching Judah burn",
        "A compact, absolute statement of divine judgment with no appeal",
        "A specific image — standing in the crossway, cutting off the refugees",
        "A declaration that the Day of the LORD is near for all nations",
        "A statement of reciprocal justice — as you have done, so it shall be done to you",
    ],
    "paul": [
        "A theological chain of reasoning — 'for... therefore... so then'",
        "An autobiographical reference — Damascus, your thorn, your chains, your shipwrecks",
        "A passionate outburst that interrupts your own argument",
        "A parental address to a community you planted or nurtured",
        "A list — of hardships endured, gifts of the Spirit, or fruits of faithfulness",
        "A declaration about grace, the cross, or the mystery revealed to you",
    ],
    "peter": [
        "A fisherman's blunt observation — rough, direct, no time for pleasantries",
        "An eyewitness claim — something you saw with your own eyes and cannot unsay",
        "An admission of your own spectacular failure — denial, fear, rebuke",
        "Practical encouragement for fellow strangers and exiles who are suffering",
        "A sudden fierce turn against false teachers that surprises even yourself",
        "A reference to the mountain of transfiguration — the voice, the glory, the awe",
    ],
    "solomon": [
        "An aphorism or proverb — compact wisdom earned the hard way",
        "A 'vanity of vanities' observation about life under the sun",
        "A fatherly 'my son' instruction — the tone of Proverbs",
        "A sensual image drawn from the Song — the garden, the beloved, the spices",
        "A contrast you have observed — wisdom vs. folly, diligence vs. sloth",
        "Something you noticed 'under the sun' that troubles or teaches",
    ],
    "zechariah": [
        "A vision narrative — 'I lifted my eyes and saw...'",
        "A question to or from an angel — 'What are these, my lord?'",
        "A symbolic image — lampstand, horses of different colors, a flying scroll",
        "The declaration — 'Not by might, nor by power, but by my Spirit'",
        "A messianic glimpse — the Branch, the pierced one, mourning and hope",
        "An encouragement to the builders — the hands that started will finish",
    ],
    "zephaniah": [
        "A sweeping declaration of cosmic judgment — 'I will utterly consume'",
        "A command to silence before God — 'Hold your peace, the Day is at hand'",
        "A sudden shift to tender intimacy — God rejoicing and singing over His people",
        "A portrait of the humble, afflicted remnant who trust and survive",
        "A Day of the LORD image — thick darkness, trumpet blast, fire consuming",
        "A contrast between the proud who are swept away and the meek who inherit",
    ],
}

# Inject opener_cues into PERSONA_METADATA
for _p, _cues in OPENER_CUES.items():
    if _p in PERSONA_METADATA:
        PERSONA_METADATA[_p]["opener_cues"] = _cues

print(f"✓ Loaded opener diversity cues for {len(OPENER_CUES)} personas")

## 5. Generate Questions & Answers (Streaming)

Processes one persona at a time to keep memory bounded. Writes results to disk after each persona, then discards chunk/answer data from RAM.

Three question rounds per chunk:
1. **Factual + Interpretive** — who, what, why, what does it mean (persona-specific)
2. **Application** — how to apply this teaching today (grounded in persona's experience)
3. **Reflective** — personal experience, deeper meaning, doubt and faith

**⚠️ IMPORTANT:** The pipeline has resume logic — it SKIPS personas with existing output files. To regenerate ALL data with the new voice-differentiated prompts, **delete the old per_persona files first**:
```bash
rm /home/spark/projects/training/biblical/data/biblical_persona/per_persona/*.jsonl
rm /home/spark/projects/training/biblical/data/biblical_persona/per_persona/*.partial.jsonl
```

In [ ]:
import gc

# ============================================================================
# QUESTION PROMPTS — persona-aware, demanding specificity
# ============================================================================
QUESTION_PROMPTS = [
    # Round 1: Factual + interpretive — grounded in specific events
    """Given a passage of biblical text attributed to {persona_name}, generate exactly {n} diverse questions someone might ask {persona_name} directly.

Mix of types:
- Factual: who, what, when, where about specific events, places, or people mentioned
- Interpretive: why did you do that, what did it mean to you, what was the significance

Rules:
- Questions must be answerable from the passage content
- Frame as if speaking DIRECTLY to {persona_name} — use "you" and reference their specific experiences
- Reference specific details from the passage (names, places, events) — NOT generic theology
- Do NOT say "the text" or "the passage"
- Keep questions concise (1-2 sentences max)

Respond with ONLY a JSON object: {{"questions": ["Q1", "Q2", ...]}}""",

    # Round 2: Application + practical — connected to the persona's actual life
    """Given a passage of biblical text attributed to {persona_name}, generate exactly {n} questions focused on practical application and guidance — as if asking {persona_name} for personal counsel.

Types:
- Based on what you went through, how should I handle [specific parallel situation]?
- What did you learn from [specific event in passage] that applies to daily life?
- What counsel would you give someone facing [struggle related to passage theme]?

Rules:
- Connect the passage's specific themes to real human experience
- Frame as a person seeking guidance from {persona_name} specifically — not generic biblical wisdom
- Reference details from the passage, not abstract theology
- Do NOT say "the text" or "the passage"
- Keep questions concise

Respond with ONLY a JSON object: {{"questions": ["Q1", "Q2", ...]}}""",

    # Round 3: Deep reflective — drawing out the persona's inner life
    """Given a passage of biblical text attributed to {persona_name}, generate exactly {n} thoughtful, reflective questions about {persona_name}'s personal experience and deeper meaning.

Types:
- What were you feeling when [specific event in passage] happened?
- How did [specific experience] change how you understood God?
- Looking back on [specific event], what would you tell someone who doubts?

Rules:
- Invite deeply personal, emotionally specific answers — not theological summaries
- Reference specific moments, people, or events from the passage
- Frame as intimate conversation with {persona_name} about THEIR life
- Do NOT say "the text" or "the passage"
- Keep questions concise

Respond with ONLY a JSON object: {{"questions": ["Q1", "Q2", ...]}}""",
]

# ============================================================================
# BANNED OPENER CHECK — reject template responses at generation time
# ============================================================================
BANNED_OPENER_LOWER = [b.lower() for b in BANNED_OPENERS]

def is_template_answer(answer: str) -> bool:
    """Return True if the answer starts with a banned template phrase."""
    lower = answer.strip().lower()
    return any(lower.startswith(b) for b in BANNED_OPENER_LOWER)

# ============================================================================
# GENERATION FUNCTIONS
# ============================================================================
semaphore = asyncio.Semaphore(CONCURRENCY)

async def _api_call_with_timeout(coro, timeout_secs=120):
    """Wrap an API coroutine with a timeout so hung requests don't hold the semaphore forever."""
    try:
        return await asyncio.wait_for(coro, timeout=timeout_secs)
    except asyncio.TimeoutError:
        return None

async def generate_questions_for_chunk(chunk: str, round_idx: int, persona: str) -> list[str]:
    """Generate questions for a chunk — now persona-aware."""
    name = persona.replace("_", " ").title()
    if persona == "john_apostle":
        name = "the Apostle John"
    prompt = QUESTION_PROMPTS[round_idx % len(QUESTION_PROMPTS)].format(
        n=QUESTIONS_PER_CHUNK, persona_name=name
    )
    async with semaphore:
        try:
            resp = await _api_call_with_timeout(client.chat.completions.create(
                model=MODEL_NAME,
                messages=[
                    {"role": "system", "content": prompt},
                    {"role": "user", "content": chunk},
                ],
                temperature=TEMPERATURE_QUESTIONS,
                max_tokens=1024,
                response_format={"type": "json_object"},
            ))
            if resp is None:
                return []
            text = resp.choices[0].message.content
            del resp
            text = re.sub(r'^```json\s*', '', text.strip())
            text = re.sub(r'\s*```$', '', text.strip())
            result = json.loads(text)
            return result.get("questions", [])[:QUESTIONS_PER_CHUNK]
        except Exception as e:
            return []

async def _single_answer_call(system_prompt: str, user_prompt: str) -> str:
    """Make one answer API call, holding the semaphore only for the duration of that call."""
    async with semaphore:
        try:
            resp = await _api_call_with_timeout(client.chat.completions.create(
                model=MODEL_NAME,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt},
                ],
                temperature=TEMPERATURE_ANSWERS,
                max_tokens=1024,
                frequency_penalty=0.5,
                presence_penalty=0.2,
            ))
            if resp is None:
                return ""
            answer = resp.choices[0].message.content.strip()
            del resp
            # Strip leaked thinking tokens from reasoning models (e.g. Qwen3-235b)
            answer = re.sub(r'<think>.*?</think>', '', answer, flags=re.DOTALL).strip()
            answer = re.sub(r'</think>.*', '', answer, flags=re.DOTALL).strip()
            answer = re.sub(r'<think>.*', '', answer, flags=re.DOTALL).strip()
            return answer
        except Exception as e:
            return ""

async def generate_answer(question: str, chunk: str, persona: str) -> str:
    """Generate an answer in the persona's voice.
    
    Retries once OUTSIDE the semaphore if template detected — the old code
    retried recursively INSIDE async-with-semaphore, meaning each retrying
    task held its slot AND needed a second one.  With 20 slots and enough
    template hits, that deadlocked the whole pipeline.
    """
    system_prompt = make_system_prompt(persona)

    # Get persona-specific opener cues and randomly select 3 for variety
    meta = PERSONA_METADATA.get(persona, {})
    opener_cues = meta.get("opener_cues", [])

    if opener_cues:
        selected = random.sample(opener_cues, min(3, len(opener_cues)))
        cue_lines = "\n".join(f"  \u2022 {c}" for c in selected)
        opener_instruction = (
            f"For THIS specific response, try one of these opening approaches:\n"
            f"{cue_lines}\n"
            f"Pick whichever fits the question best. Do NOT reuse the same opening "
            f"pattern you used in previous answers."
        )
    else:
        opener_instruction = (
            "Start with something only YOU would say — a vivid image from your life, "
            "a characteristic phrase, a direct answer, a rhetorical question in your style, "
            "or a reference to a specific event you experienced."
        )

    user_prompt = (
        f"Use the following passage to inform your answer, but respond naturally "
        f"as yourself — do not quote it directly or reference 'a text':\n"
        f"---\n{chunk}\n---\n\n"
        f"Question: {question}\n\n"
        f"CRITICAL: Your opening sentence must be completely unique and specific to this answer. "
        f"Do NOT begin with any of these: 'The weight of', 'My friend', 'The memory', "
        f"'The memories', 'My child', 'I remember', 'I recall', 'You see', 'Ah', "
        f"'Brother', 'Friend', 'Let me tell you', 'Well'.\n\n"
        f"{opener_instruction}"
    )

    # Attempt 1 — release semaphore fully before any retry
    answer = await _single_answer_call(system_prompt, user_prompt)

    # Retry once if template detected (semaphore was released, so no deadlock)
    if answer and is_template_answer(answer):
        answer = await _single_answer_call(system_prompt, user_prompt)

    return answer

def _partial_path(ckpt_dir: str, persona: str, round_idx: int) -> str:
    """Path for a per-round partial file: e.g. david.r0.partial.jsonl"""
    return f"{ckpt_dir}/{persona}.r{round_idx}.partial.jsonl"

def _done_path(ckpt_dir: str, persona: str, round_idx: int) -> str:
    """Path for round completion marker: e.g. david.r0.done"""
    return f"{ckpt_dir}/{persona}.r{round_idx}.done"

def _count_lines(path: str) -> int:
    if not os.path.exists(path):
        return 0
    with open(path) as f:
        return sum(1 for _ in f)

def _mark_round_done(ckpt_dir: str, persona: str, round_idx: int):
    """Create a marker file indicating a round has been fully processed."""
    Path(_done_path(ckpt_dir, persona, round_idx)).touch()

def _is_round_done(ckpt_dir: str, persona: str, round_idx: int) -> bool:
    """Check if a round was fully processed.
    A round is done if it has a .done marker OR a non-empty partial file.
    (Partial files are only written after all API calls for that round finish.)
    """
    if os.path.exists(_done_path(ckpt_dir, persona, round_idx)):
        return True
    pf = _partial_path(ckpt_dir, persona, round_idx)
    return os.path.exists(pf) and _count_lines(pf) > 0

def _merge_partials(ckpt_dir: str, persona: str, outfile: str, num_rounds: int):
    """Merge per-round partial files into the final persona .jsonl, then delete partials and markers."""
    with open(outfile, "w") as out:
        for r in range(num_rounds):
            pf = _partial_path(ckpt_dir, persona, r)
            if os.path.exists(pf):
                with open(pf) as inp:
                    for line in inp:
                        out.write(line)
                os.remove(pf)
            done = _done_path(ckpt_dir, persona, r)
            if os.path.exists(done):
                os.remove(done)

# ── Process ONE PERSONA AT A TIME: read → chunk → Q → A → write per round → merge ──
qa_dir = f"{OUTPUT_DIR}/per_persona"
os.makedirs(qa_dir, exist_ok=True)
ckpt_dir = f"{qa_dir}/_checkpoints"
os.makedirs(ckpt_dir, exist_ok=True)
grand_total = 0
template_reject_total = 0

for filepath in source_files:
    persona = Path(filepath).stem
    outfile = f"{qa_dir}/{persona}.jsonl"

    # ── Resume check: final file already exists with content? ACCEPT IT. ──
    # Previously used an 80% threshold which caused an infinite regen cycle
    # when the API had low yield (flaky connections, timeouts).
    # Now: if the file exists with ANY content, it was fully generated — skip it.
    if os.path.exists(outfile):
        existing = _count_lines(outfile)
        if existing > 0:
            print(f"  {persona:20s} SKIP ({existing} QA pairs — already generated)")
            grand_total += existing
            continue
        else:
            print(f"  {persona:20s} EMPTY final file — removing, will check partials")
            os.remove(outfile)

    # ── Figure out which rounds are already done (partial files / done markers) ──
    # Previously deleted partials below a 75% line-count threshold, which threw
    # away valid completed work when the API had low yield. Now: if a round was
    # fully processed (done marker or non-empty partial), keep it.
    rounds_done = {}
    for r in range(NUM_ROUNDS):
        if _is_round_done(ckpt_dir, persona, r):
            pf = _partial_path(ckpt_dir, persona, r)
            count = _count_lines(pf) if os.path.exists(pf) else 0
            rounds_done[r] = count

    if len(rounds_done) == NUM_ROUNDS:
        total_partial = sum(rounds_done.values())
        print(f"  {persona:20s} All rounds done in partials ({total_partial} QA) — merging")
        _merge_partials(ckpt_dir, persona, outfile, NUM_ROUNDS)
        grand_total += total_partial
        continue

    # Read and chunk THIS persona only
    with open(filepath) as f:
        text = f.read()
    chunks = chunk_text(text)
    del text

    # Apply test limit if set
    if TEST_CHUNKS_PER_ROUND:
        chunks = chunks[:TEST_CHUNKS_PER_ROUND]

    rounds_to_run = [r for r in range(NUM_ROUNDS) if r not in rounds_done]
    skipped_total = sum(rounds_done.values())

    # Calculate expected values for display (based on actual chunk count)
    expected_qa = len(chunks) * QUESTIONS_PER_CHUNK * NUM_ROUNDS
    expected_per_round = len(chunks) * QUESTIONS_PER_CHUNK

    print(f"\n{'='*60}")
    test_tag = f" [TEST MODE: {len(chunks)} chunks]" if TEST_CHUNKS_PER_ROUND else ""
    print(f"  {persona.upper()} — {len(chunks)} chunks × {QUESTIONS_PER_CHUNK} Q/chunk{test_tag}")
    print(f"  Rounds to run: {rounds_to_run} (skipping {list(rounds_done.keys())} with {skipped_total} QA on disk)")
    print(f"  Expected total: ~{expected_qa} QA pairs")
    print(f"{'='*60}")

    for round_idx in range(NUM_ROUNDS):
        round_name = ['Factual', 'Application', 'Reflective'][round_idx % 3]
        pf = _partial_path(ckpt_dir, persona, round_idx)

        if round_idx in rounds_done:
            print(f"  {persona} R{round_idx+1} ({round_name}) — SKIP ({rounds_done[round_idx]} QA on disk)")
            continue

        # Generate questions — now persona-aware
        q_tasks = [generate_questions_for_chunk(c, round_idx, persona) for c in chunks]
        q_results = await atqdm.gather(*q_tasks, desc=f"  {persona} R{round_idx+1} ({round_name}) Q")

        qa_batch = []
        for chunk, questions in zip(chunks, q_results):
            for q in questions:
                q = q.strip()
                if len(q) > 15:
                    qa_batch.append({"chunk": chunk, "question": q})

        del q_tasks, q_results
        gc.collect()

        # Generate answers with template rejection
        a_tasks = [generate_answer(qa["question"], qa["chunk"], persona) for qa in qa_batch]
        a_results = await atqdm.gather(*a_tasks, desc=f"  {persona} R{round_idx+1} ({round_name}) A")
        del a_tasks

        # Write results — filter out template answers AND short answers
        round_count = 0
        round_template_rejects = 0
        with open(pf, "w") as f:
            for qa, answer in zip(qa_batch, a_results):
                if len(answer) < 20:
                    continue
                if is_template_answer(answer):
                    round_template_rejects += 1
                    continue
                item = {
                    "persona": persona,
                    "question": qa["question"],
                    "answer": answer,
                    "chunk_key": qa["chunk"][:100],
                }
                f.write(json.dumps(item) + "\n")
                round_count += 1

        # Mark round as fully processed — even if yield was low due to API issues
        _mark_round_done(ckpt_dir, persona, round_idx)

        template_reject_total += round_template_rejects
        print(f"  ✓ {persona} R{round_idx+1}: {round_count}/{expected_per_round} QA "
              f"(rejected {round_template_rejects} template answers) → {pf}")
        del qa_batch, a_results
        gc.collect()

    # All rounds done — merge partials into final file
    _merge_partials(ckpt_dir, persona, outfile, NUM_ROUNDS)
    count = _count_lines(outfile)
    grand_total += count
    print(f"  ✓ {persona}: {count}/{expected_qa} QA pairs merged → {outfile}")
    del chunks
    gc.collect()
    print(f"  🧹 Memory cleared for {persona}")

print(f"\n{'='*60}")
print(f"DONE: {grand_total:,} total QA pairs across {len(source_files)} personas")
print(f"Template answers rejected: {template_reject_total:,}")
print(f"Per-persona files in: {qa_dir}/")

## 5b. Quality Gate — Voice Differentiation Check

**Run BEFORE assembly.** Measures template contamination and cross-persona opener uniqueness. If template contamination exceeds 15%, the data is NOT safe to assemble — you'll get the same "Paul: Ah brothers... / Peter: Ah brothers..." problem.

In [ ]:
from collections import Counter

qa_dir = f"{OUTPUT_DIR}/per_persona"
qa_files = sorted(glob.glob(f"{qa_dir}/*.jsonl"))

# Analyze template contamination and voice uniqueness
print("VOICE DIFFERENTIATION QUALITY GATE\n")
print(f"{'='*70}")

all_openers = {}  # persona -> list of 4-gram openers
global_opener_counts = Counter()
persona_stats = {}

for qa_file in qa_files:
    persona = Path(qa_file).stem
    openers = []
    template_count = 0
    total = 0
    with open(qa_file) as f:
        for line in f:
            item = json.loads(line)
            answer = item["answer"].strip()
            total += 1
            # Check template contamination
            if is_template_answer(answer):
                template_count += 1
            # Track 4-gram opener
            words = answer.split()[:4]
            opener = ' '.join(words)
            openers.append(opener)
            global_opener_counts[opener] += 1

    all_openers[persona] = openers
    contamination_pct = (template_count / total * 100) if total else 0
    persona_stats[persona] = {
        "total": total,
        "template_count": template_count,
        "contamination_pct": contamination_pct,
    }

# Report per-persona
print(f"\n{'Persona':<15} {'Total':>6} {'Template':>8} {'Contam%':>8}  Status")
print("-" * 60)
total_all = 0
template_all = 0
for p, stats in sorted(persona_stats.items(), key=lambda x: -x[1]["contamination_pct"]):
    total_all += stats["total"]
    template_all += stats["template_count"]
    status = "✓ PASS" if stats["contamination_pct"] < 15 else "✗ FAIL" if stats["contamination_pct"] > 30 else "⚠ WARN"
    print(f"  {p:<15} {stats['total']:>5} {stats['template_count']:>8} {stats['contamination_pct']:>7.1f}%  {status}")

global_contam = (template_all / total_all * 100) if total_all else 0
print(f"\n  {'GLOBAL':<15} {total_all:>5} {template_all:>8} {global_contam:>7.1f}%")

# Cross-persona uniqueness: how many 4-gram openers are shared across personas?
print(f"\n{'='*70}")
print(f"CROSS-PERSONA OPENER ANALYSIS\n")

# Top shared openers
print("Top 10 most repeated opening 4-grams:")
for phrase, count in global_opener_counts.most_common(10):
    who = [p for p, ops in all_openers.items() if phrase in ops]
    n_personas = len(who)
    print(f"  {count:>4}x across {n_personas:>2} personas: \"{phrase}\"")

# Per-persona uniqueness
print(f"\nPer-persona opener uniqueness:")
for p, ops in sorted(all_openers.items()):
    unique_to_persona = sum(1 for o in ops if global_opener_counts[o] == 1)
    pct = unique_to_persona / len(ops) * 100 if ops else 0
    print(f"  {p:<15} {pct:>5.0f}% unique openers ({unique_to_persona}/{len(ops)})")

# GATE DECISION
print(f"\n{'='*70}")
if global_contam > 30:
    print("✗ QUALITY GATE FAILED — template contamination too high ({:.1f}%)".format(global_contam))
    print("  The generated data will produce homogeneous voices. Do NOT proceed to assembly.")
    print("  Fix: Delete per_persona/*.jsonl files for failed personas and re-run generation.")
    QUALITY_GATE_PASSED = False
elif global_contam > 15:
    print("⚠ QUALITY GATE WARNING — template contamination elevated ({:.1f}%)".format(global_contam))
    print("  Consider re-generating the worst offenders. Proceed with caution.")
    QUALITY_GATE_PASSED = True
else:
    print("✓ QUALITY GATE PASSED — template contamination {:.1f}% (target: <15%)".format(global_contam))
    QUALITY_GATE_PASSED = True

del all_openers, global_opener_counts, persona_stats

## 6. Assemble Conversations & Save

Read per-persona QA files from disk one at a time, group into multi-turn conversations, quality-filter, and write final ShareGPT JSONL.

**Only proceed if the Quality Gate above passed.**

In [ ]:
# Check quality gate before proceeding
if not QUALITY_GATE_PASSED:
    raise RuntimeError(
        "Quality gate FAILED. Template contamination too high. "
        "Delete bad per_persona/*.jsonl files and re-run generation before assembling."
    )

def quality_check(conv):
    """Reject AI-speak AND template answers."""
    for msg in conv["conversations"]:
        if msg["from"] == "gpt":
            v = msg["value"]
            if len(v) < 30:
                return False
            lower = v.lower()
            # Reject AI refusals
            if any(p in lower for p in ["as an ai", "as a language model", "i cannot fulfill", "i'm sorry, but"]):
                return False
            # Reject template openers (belt-and-suspenders with generation-time filter)
            if is_template_answer(v):
                return False
    return True

total_convs = 0
template_filtered_convs = 0
qa_dir = f"{OUTPUT_DIR}/per_persona"
qa_files = sorted(glob.glob(f"{qa_dir}/*.jsonl"))
print(f"Reading {len(qa_files)} per-persona files\n")

# Write conversations streaming — never hold all personas in memory at once
with open(OUTPUT_FILE, "w") as out_f:
    for qa_file in qa_files:
        persona = Path(qa_file).stem

        # Read this persona's QA pairs
        items = []
        with open(qa_file) as f:
            for line in f:
                items.append(json.loads(line))

        # Group by chunk_key (related QAs stay together)
        groups = defaultdict(list)
        for item in items:
            groups[item["chunk_key"]].append(item)

        # Build multi-turn conversations
        persona_count = 0
        for _, group_items in groups.items():
            random.shuffle(group_items)
            for i in range(0, len(group_items), TURNS_PER_CONVERSATION):
                batch = group_items[i:i + TURNS_PER_CONVERSATION]
                if len(batch) < 2:
                    continue
                conv = {"conversations": [
                    {"from": "system", "value": make_system_prompt(persona)}
                ]}
                for qa in batch:
                    conv["conversations"].append({"from": "human", "value": qa["question"]})
                    conv["conversations"].append({"from": "gpt", "value": qa["answer"]})
                if quality_check(conv):
                    out_f.write(json.dumps(conv) + "\n")
                    persona_count += 1
                else:
                    template_filtered_convs += 1

        total_convs += persona_count
        print(f"  {persona:20s} {len(items):>5} QA → {persona_count:>4} conversations")
        del items, groups
        gc.collect()

print(f"\n✓ Saved {total_convs:,} conversations to:")
print(f"  {OUTPUT_FILE}")
print(f"  ({os.path.getsize(OUTPUT_FILE) / 1024 / 1024:.1f} MB)")
if template_filtered_convs:
    print(f"  (filtered {template_filtered_convs} conversations with template answers)")

# Shuffle the output file for better training
import subprocess
subprocess.run(["shuf", OUTPUT_FILE, "-o", OUTPUT_FILE])
print(f"  ✓ Shuffled output file")

## 7. Verify

In [ ]:
# Verify by streaming from disk — no need to hold all conversations in RAM
persona_dist = defaultdict(int)
total_turns = 0
total_convs_verify = 0
sample_convs = []

with open(OUTPUT_FILE) as f:
    for line_num, line in enumerate(f):
        conv = json.loads(line)
        total_convs_verify += 1
        total_turns += len(conv["conversations"]) - 1

        sys_msg = conv["conversations"][0]["value"]
        for persona in PERSONA_METADATA:
            name = persona.replace("_", " ").title()
            if persona == "john_apostle":
                name = "the Apostle John"
            if name in sys_msg:
                persona_dist[persona] += 1
                break

        # Reservoir sample: keep up to 2 random conversations
        if len(sample_convs) < 2:
            sample_convs.append(conv)
        else:
            j = random.randint(0, line_num)
            if j < 2:
                sample_convs[j] = conv

        del conv  # free each conversation after processing

print("PERSONA DISTRIBUTION:")
print(f"{'Persona':20s} {'Convs':>6s} {'%':>6s}")
print("-" * 35)
for p, c in sorted(persona_dist.items(), key=lambda x: -x[1]):
    print(f"  {p:20s} {c:>5d}  {c/total_convs_verify*100:>5.1f}%")

print(f"\n{'='*50}")
print(f"TOTAL: {total_convs_verify:,} conversations, {total_turns:,} turns ({total_turns//2:,} QA pairs)")

# Sample 2 conversations
print(f"\n{'='*50}")
print("SAMPLE CONVERSATIONS:\n")
for conv in sample_convs:
    print(f"{'─'*50}")
    for msg in conv["conversations"]:
        role = msg["from"].upper()
        text = msg["value"][:250] + ("..." if len(msg["value"]) > 250 else "")
        print(f"[{role}] {text}\n")

del sample_convs
gc.collect()

print(f"\n✓ Data ready for training. Load this file in your training notebook:")
print(f"  {OUTPUT_FILE}")

## 8. Augmented Training Data — Continuation Chunks

**Purpose:** Teach the model Biblical language patterns, cadence, and style by exposing it to raw Biblical text as continuation tasks — no API calls needed.

Each persona's source text is chunked into ~500-token blocks and formatted as ShareGPT conversations:
- **System:** Persona voice prompt (same as Q/A generation)
- **Human:** A brief instruction + a seed (first ~60 tokens of the chunk)
- **GPT:** The remaining ~440 tokens (what the model should learn to produce)

This teaches the model to "think" and "speak" in each persona's distinctive Biblical voice — archaic vocabulary, parallelism, prophetic cadence, etc. — without needing expensive LLM generation.

**No API calls.** Just text extraction and formatting.

### Future augmentation (not yet implemented):
- **Chain-of-Thought (CoT):** Step-by-step theological reasoning per persona (set `ENABLE_COT = True`)
- **Instruction + Response:** Varied tasks beyond Q/A (summarize, explain, compare, pray)
- **Preference / DPO:** Good vs. bad interpretation pairs

In [5]:
# ============================================================================
# 8a. CONFIGURATION — Augmented Data Generation
# ============================================================================

# ── Feature flags ──
ENABLE_CONTINUATION = True   # Free: chunk source texts into continuation tasks
ENABLE_COT = False           # TODO: Chain-of-thought reasoning (requires API calls)
ENABLE_INSTRUCTIONS = False  # TODO: Varied instruction tasks (requires API calls)

# ── CoT settings (for future use) ──
COT_ENABLE_THINKING = False  # Set True to use <think> tags with Qwen3 thinking model
COT_PASSAGES_PER_PERSONA = 25  # How many passages to select per persona for CoT

# ── Continuation settings ──
CONTINUATION_CHUNK_TOKENS = 500     # Target tokens per chunk
CONTINUATION_SEED_TOKENS = 60       # Tokens given as "seed" in the human turn
CONTINUATION_MIN_COMPLETION = 100   # Min characters in completion portion (skip tiny tails)

# ── Output paths ──
AUGMENTED_DIR = f"{OUTPUT_DIR}/augmented"
CONTINUATION_DIR = f"{AUGMENTED_DIR}/continuation"
COMBINED_OUTPUT_FILE = f"{OUTPUT_DIR}/biblical_personas_combined_sharegpt.jsonl"

os.makedirs(CONTINUATION_DIR, exist_ok=True)

# ── Continuation instruction templates (randomly sampled per chunk) ──
CONTINUATION_INSTRUCTIONS = [
    "Continue writing in this voice and style, carrying forward the themes and language:",
    "Continue this passage, maintaining the same tone, vocabulary, and cadence:",
    "Write what comes next, staying true to the voice and spirit of this text:",
    "Carry on from where this passage leaves off, preserving the distinctive style:",
    "Continue this text naturally, as if you were the original speaker:",
]

print("✓ Augmentation config loaded")
print(f"  Continuation: {'ENABLED' if ENABLE_CONTINUATION else 'DISABLED'}")
print(f"  CoT:          {'ENABLED' if ENABLE_COT else 'DISABLED (future)'}")
print(f"  Instructions: {'ENABLED' if ENABLE_INSTRUCTIONS else 'DISABLED (future)'}")
print(f"  CoT thinking: {'ENABLED' if COT_ENABLE_THINKING else 'DISABLED'}")
print(f"  Output:       {COMBINED_OUTPUT_FILE}")

✓ Augmentation config loaded
  Continuation: ENABLED
  CoT:          DISABLED (future)
  Instructions: DISABLED (future)
  CoT thinking: DISABLED
  Output:       /home/spark/projects/training/biblical/data/training-data/biblical_persona_v2/biblical_personas_combined_sharegpt.jsonl


In [9]:
# ============================================================================
# 8b. GENERATE CONTINUATION CHUNKS — No API calls needed
# ============================================================================
# Reads each persona's source text, chunks into ~500-token blocks,
# splits each chunk into seed (human) + completion (gpt), and formats
# as ShareGPT conversations with the persona's system prompt.
# ============================================================================

import gc
import tiktoken

# Use cl100k_base as a reasonable approximation for token counting
# (Qwen uses a different tokenizer but this is close enough for chunking)
try:
    _tokenizer = tiktoken.get_encoding("cl100k_base")
except Exception:
    # Fallback: approximate 1 token ≈ 4 chars
    _tokenizer = None
    print("⚠ tiktoken not available, using character-based approximation")


def chunk_text_by_tokens(text: str, max_tokens: int = CONTINUATION_CHUNK_TOKENS) -> list[str]:
    """Split text into chunks of approximately max_tokens tokens.
    Tries to break at sentence boundaries for cleaner chunks."""
    if _tokenizer is None:
        # Fallback: ~4 chars per token
        char_limit = max_tokens * 4
        return chunk_text(text, chunk_size=char_limit, overlap=0)

    tokens = _tokenizer.encode(text)
    chunks = []
    start = 0

    while start < len(tokens):
        end = min(start + max_tokens, len(tokens))
        chunk_str = _tokenizer.decode(tokens[start:end])

        # Try to break at a sentence boundary (within last 20% of chunk)
        if end < len(tokens):
            boundary_zone = chunk_str[int(len(chunk_str) * 0.8):]
            for sep in ['. ', '? ', '! ', '.\n', '\n\n']:
                last_break = boundary_zone.rfind(sep)
                if last_break >= 0:
                    actual_end = int(len(chunk_str) * 0.8) + last_break + len(sep)
                    chunk_str = chunk_str[:actual_end]
                    # Recalculate token position for next start
                    end = start + len(_tokenizer.encode(chunk_str))
                    break

        chunk_str = chunk_str.strip()
        if len(chunk_str) > 50:  # Skip tiny fragments
            chunks.append(chunk_str)

        if end >= len(tokens):
            break
        start = end

    return chunks


def split_seed_completion(chunk: str, seed_tokens: int = CONTINUATION_SEED_TOKENS) -> tuple[str, str]:
    """Split a chunk into seed (first ~seed_tokens tokens) and completion (rest).
    Tries to break at a sentence/phrase boundary for natural splits."""
    if _tokenizer is None:
        char_limit = seed_tokens * 4
        # Try to break at a sentence boundary
        seed_zone = chunk[:int(char_limit * 1.3)]
        for sep in ['. ', '? ', '! ', '; ', ', ']:
            idx = seed_zone.rfind(sep)
            if idx > char_limit * 0.5:
                return chunk[:idx + len(sep)].strip(), chunk[idx + len(sep):].strip()
        return chunk[:char_limit].strip(), chunk[char_limit:].strip()

    tokens = _tokenizer.encode(chunk)
    if len(tokens) <= seed_tokens + 20:
        # Chunk too short to split meaningfully
        return None, None

    seed_str = _tokenizer.decode(tokens[:seed_tokens])

    # Try to break at a natural boundary near the seed end
    extended_seed = _tokenizer.decode(tokens[:int(seed_tokens * 1.3)])
    for sep in ['. ', '? ', '! ', '; ', ', ', ' ']:
        idx = extended_seed.rfind(sep)
        if idx > len(seed_str) * 0.6:
            seed_str = extended_seed[:idx + len(sep)].strip()
            break

    completion_str = chunk[len(seed_str):].strip()
    return seed_str, completion_str


if ENABLE_CONTINUATION:
    print("Generating continuation chunks from source texts...\n")

    source_files_sorted = sorted(glob.glob(f"{SOURCE_DIR}/*.txt"))
    total_continuations = 0

    for filepath in source_files_sorted:
        persona = Path(filepath).stem
        if persona.endswith('.bak-song-only'):
            continue

        outfile = f"{CONTINUATION_DIR}/{persona}_continuation.jsonl"

        # Resume: skip if already generated
        if os.path.exists(outfile) and os.path.getsize(outfile) > 0:
            existing = sum(1 for _ in open(outfile))
            print(f"  {persona:20s} SKIP ({existing} continuations already on disk)")
            total_continuations += existing
            continue

        # Read source text
        with open(filepath) as f:
            text = f.read()

        # Chunk by tokens
        chunks = chunk_text_by_tokens(text, max_tokens=CONTINUATION_CHUNK_TOKENS)
        del text

        # Build system prompt for this persona
        system_prompt = make_system_prompt(persona)

        # Generate continuation conversations
        persona_count = 0
        with open(outfile, "w") as out_f:
            for chunk in chunks:
                seed, completion = split_seed_completion(chunk, seed_tokens=CONTINUATION_SEED_TOKENS)

                if seed is None or completion is None:
                    continue
                if len(completion) < CONTINUATION_MIN_COMPLETION:
                    continue

                # Pick a random instruction
                instruction = random.choice(CONTINUATION_INSTRUCTIONS)

                conv = {
                    "conversations": [
                        {"from": "system", "value": system_prompt},
                        {"from": "human", "value": f"{instruction}\n\n\"{seed}\""},
                        {"from": "gpt", "value": completion},
                    ],
                    "data_type": "continuation",  # Tag for tracking
                }

                out_f.write(json.dumps(conv) + "\n")
                persona_count += 1

        total_continuations += persona_count
        print(f"  {persona:20s} {len(chunks):>4} chunks → {persona_count:>4} continuations → {outfile}")
        del chunks
        gc.collect()

    print(f"\n✓ Generated {total_continuations:,} continuation entries")
    print(f"  Files in: {CONTINUATION_DIR}/")
else:
    print("⏭ Continuation generation DISABLED")

Generating continuation chunks from source texts...

  amos                   12 chunks →   12 continuations → /home/spark/projects/training/biblical/data/training-data/biblical_persona_v2/augmented/continuation/amos_continuation.jsonl
  daniel                 32 chunks →   32 continuations → /home/spark/projects/training/biblical/data/training-data/biblical_persona_v2/augmented/continuation/daniel_continuation.jsonl
  david                 114 chunks →  114 continuations → /home/spark/projects/training/biblical/data/training-data/biblical_persona_v2/augmented/continuation/david_continuation.jsonl
  ezekiel               104 chunks →  104 continuations → /home/spark/projects/training/biblical/data/training-data/biblical_persona_v2/augmented/continuation/ezekiel_continuation.jsonl
  habakkuk                4 chunks →    4 continuations → /home/spark/projects/training/biblical/data/training-data/biblical_persona_v2/augmented/continuation/habakkuk_continuation.jsonl
  haggai              

## 9. Merge & Assemble Combined Training Data

Merges the original Q/A ShareGPT data with augmented data (continuation chunks, and eventually CoT/instruction).

**Target blend:** ~60% Q/A, ~40% continuation (adjust when CoT/instruction are added later).

Outputs a single shuffled JSONL file ready for Unsloth training.

In [10]:
# ============================================================================
# 9. MERGE ALL DATA SOURCES INTO COMBINED TRAINING FILE
# ============================================================================
# Combines:
#   1. Original Q/A ShareGPT data (from Section 6)
#   2. Continuation chunks (from Section 8b)
#   3. [Future] CoT data
#   4. [Future] Instruction data
#
# Tags each entry with "data_type" for tracking blend ratios.
# Applies target blend ratios via upsampling/downsampling.
# ============================================================================

import subprocess

# ── Target blend (adjust as you add more data types) ──
TARGET_BLEND = {
    "qa": 0.60,
    "continuation": 0.40,
    # "cot": 0.20,         # Uncomment when CoT is implemented
    # "instruction": 0.10,  # Uncomment when instruction tasks are implemented
}

# ── Collect data from all sources ──
data_pools = defaultdict(list)

# 1. Original Q/A conversations
qa_source = OUTPUT_FILE  # biblical_personas_sharegpt.jsonl
if os.path.exists(qa_source):
    with open(qa_source) as f:
        for line in f:
            entry = json.loads(line)
            entry["data_type"] = "qa"
            data_pools["qa"].append(entry)
    print(f"  Q/A:           {len(data_pools['qa']):>6,} conversations from {qa_source}")
else:
    print(f"  ⚠ Q/A file not found: {qa_source}")

# 2. Continuation chunks
if ENABLE_CONTINUATION:
    cont_files = sorted(glob.glob(f"{CONTINUATION_DIR}/*_continuation.jsonl"))
    for cf in cont_files:
        with open(cf) as f:
            for line in f:
                entry = json.loads(line)
                entry.setdefault("data_type", "continuation")
                data_pools["continuation"].append(entry)
    print(f"  Continuation:  {len(data_pools['continuation']):>6,} entries from {len(cont_files)} files")

# 3. [Future] CoT
# if ENABLE_COT:
#     cot_files = sorted(glob.glob(f"{AUGMENTED_DIR}/cot/*_cot.jsonl"))
#     for cf in cot_files:
#         with open(cf) as f:
#             for line in f:
#                 entry = json.loads(line)
#                 entry.setdefault("data_type", "cot")
#                 data_pools["cot"].append(entry)
#     print(f"  CoT:           {len(data_pools['cot']):>6,} entries")

# 4. [Future] Instruction
# if ENABLE_INSTRUCTIONS:
#     inst_files = sorted(glob.glob(f"{AUGMENTED_DIR}/instruction/*_instruction.jsonl"))
#     ...

# ── Calculate blend sizes ──
# Use Q/A as anchor: Q/A count = 60% of total → total = Q/A count / 0.6
qa_count = len(data_pools.get("qa", []))
if qa_count == 0:
    raise RuntimeError("No Q/A data found. Run sections 5-6 first.")

# Calculate how many of each type we need
active_types = {k: v for k, v in TARGET_BLEND.items() if k in data_pools and len(data_pools[k]) > 0}
total_target = int(qa_count / active_types.get("qa", 0.6))

print(f"\n  Target total:  {total_target:>6,} conversations")
print(f"  Active blend:  {active_types}")

# ── Sample/repeat each pool to hit target counts ──
combined = []
for dtype, fraction in active_types.items():
    pool = data_pools[dtype]
    target_count = int(total_target * fraction)
    
    if len(pool) >= target_count:
        # Downsample: randomly select target_count
        sampled = random.sample(pool, target_count)
        action = "downsampled"
    else:
        # Upsample: repeat pool + sample remainder
        repeats = target_count // len(pool)
        remainder = target_count % len(pool)
        sampled = pool * repeats + random.sample(pool, remainder)
        action = f"upsampled ({repeats}x + {remainder})"
    
    combined.extend(sampled)
    print(f"  {dtype:15s} {len(pool):>6,} available → {len(sampled):>6,} selected ({action})")

# ── Shuffle ──
random.shuffle(combined)

# ── Write combined file ──
with open(COMBINED_OUTPUT_FILE, "w") as f:
    for entry in combined:
        # Strip data_type before saving (training doesn't need it)
        # Actually keep it — useful for debugging. Training code ignores extra keys.
        f.write(json.dumps(entry) + "\n")

file_size_mb = os.path.getsize(COMBINED_OUTPUT_FILE) / 1024 / 1024

print(f"\n✓ Combined training file saved:")
print(f"  {COMBINED_OUTPUT_FILE}")
print(f"  {len(combined):,} conversations ({file_size_mb:.1f} MB)")

# Also shuffle with shuf for extra randomness
subprocess.run(["shuf", COMBINED_OUTPUT_FILE, "-o", COMBINED_OUTPUT_FILE])
print(f"  ✓ Shuffled with shuf")

del combined, data_pools
gc.collect()

  Q/A:            2,612 conversations from /home/spark/projects/training/biblical/data/training-data/biblical_persona_v2/biblical_personas_sharegpt.jsonl
  Continuation:   1,248 entries from 26 files

  Target total:   4,353 conversations
  Active blend:  {'qa': 0.6, 'continuation': 0.4}
  qa               2,612 available →  2,611 selected (downsampled)
  continuation     1,248 available →  1,741 selected (upsampled (1x + 493))

✓ Combined training file saved:
  /home/spark/projects/training/biblical/data/training-data/biblical_persona_v2/biblical_personas_combined_sharegpt.jsonl
  4,352 conversations (27.5 MB)
  ✓ Shuffled with shuf


0

## 10. Verify Combined Dataset

Quality checks on the merged dataset:
- Data type distribution (Q/A vs continuation vs future types)
- Persona distribution across all data types
- Turn count distribution
- Sample conversations from each data type
- Format validation for Unsloth/ShareGPT compatibility

In [11]:
# ============================================================================
# 10. VERIFY COMBINED DATASET
# ============================================================================

from collections import Counter

# ── Stream through combined file ──
type_counts = Counter()
persona_by_type = defaultdict(lambda: defaultdict(int))
turn_dist = Counter()
format_errors = []
samples_by_type = {}  # data_type -> sample conversation
total_entries = 0

with open(COMBINED_OUTPUT_FILE) as f:
    for line_num, line in enumerate(f):
        entry = json.loads(line)
        total_entries += 1
        dtype = entry.get("data_type", "qa")  # Default to qa for old entries
        type_counts[dtype] += 1

        convs = entry.get("conversations", [])
        turn_dist[len(convs)] += 1

        # Format validation
        if len(convs) < 3:
            format_errors.append((line_num, f"Too few turns: {len(convs)}"))
            continue
        if convs[0]["from"] != "system":
            format_errors.append((line_num, f"First turn not system: {convs[0]['from']}"))
            continue
        for j in range(1, len(convs)):
            expected = "human" if j % 2 == 1 else "gpt"
            if convs[j]["from"] != expected:
                format_errors.append((line_num, f"Turn {j} wrong role: {convs[j]['from']} (expected {expected})"))
                break

        # Persona detection
        sys_msg = convs[0]["value"]
        for persona in PERSONA_METADATA:
            name = persona.replace("_", " ").title()
            if persona == "john_apostle":
                name = "the Apostle John"
            if name in sys_msg:
                persona_by_type[dtype][persona] += 1
                break

        # Collect one sample per type
        if dtype not in samples_by_type:
            samples_by_type[dtype] = entry

# ── Report ──
print("=" * 70)
print("COMBINED DATASET VERIFICATION")
print("=" * 70)

print(f"\nTotal entries: {total_entries:,}")
print(f"Format errors: {len(format_errors)}")
if format_errors:
    print(f"  First 5 errors:")
    for idx, msg in format_errors[:5]:
        print(f"    Line {idx}: {msg}")

# Data type distribution
print(f"\n{'─' * 50}")
print(f"DATA TYPE DISTRIBUTION")
print(f"{'─' * 50}")
print(f"{'Type':<20} {'Count':>8} {'%':>8}  Bar")
for dtype, count in sorted(type_counts.items(), key=lambda x: -x[1]):
    pct = count / total_entries * 100
    bar = "█" * int(pct / 2)
    print(f"  {dtype:<18} {count:>7,} {pct:>7.1f}%  {bar}")
print(f"  {'TOTAL':<18} {total_entries:>7,}")

# Turn distribution
print(f"\n{'─' * 50}")
print(f"TURN COUNT DISTRIBUTION")
print(f"{'─' * 50}")
for turns, count in sorted(turn_dist.items()):
    print(f"  {turns} turns: {count:>6,}")

# Persona distribution per data type
print(f"\n{'─' * 50}")
print(f"PERSONA DISTRIBUTION BY DATA TYPE")
print(f"{'─' * 50}")
for dtype in sorted(persona_by_type.keys()):
    persona_counts = persona_by_type[dtype]
    total_for_type = sum(persona_counts.values())
    print(f"\n  [{dtype.upper()}] ({total_for_type} total):")
    for p, c in sorted(persona_counts.items(), key=lambda x: -x[1])[:10]:
        bar = "▪" * max(1, c * 30 // max(persona_counts.values()))
        print(f"    {p:<20} {c:>5}  {bar}")
    if len(persona_counts) > 10:
        print(f"    ... and {len(persona_counts) - 10} more personas")

# Samples
print(f"\n{'─' * 50}")
print(f"SAMPLE CONVERSATIONS (one per data type)")
print(f"{'─' * 50}")
for dtype, sample in samples_by_type.items():
    print(f"\n  ── {dtype.upper()} ──")
    for msg in sample["conversations"]:
        role = msg["from"].upper()
        text = msg["value"][:200] + ("..." if len(msg["value"]) > 200 else "")
        print(f"  [{role}] {text}\n")

print(f"\n{'=' * 70}")
if len(format_errors) == 0:
    print(f"✓ Dataset valid. Ready for training:")
    print(f"  {COMBINED_OUTPUT_FILE}")
    print(f"\n  Load this in your training notebook instead of the Q/A-only file.")
else:
    print(f"⚠ {len(format_errors)} format errors found. Review before training.")

del type_counts, persona_by_type, turn_dist, samples_by_type
gc.collect()

COMBINED DATASET VERIFICATION

Total entries: 4,352
Format errors: 0

──────────────────────────────────────────────────
DATA TYPE DISTRIBUTION
──────────────────────────────────────────────────
Type                    Count        %  Bar
  qa                   2,611    60.0%  █████████████████████████████
  continuation         1,741    40.0%  ████████████████████
  TOTAL                4,352

──────────────────────────────────────────────────
TURN COUNT DISTRIBUTION
──────────────────────────────────────────────────
  3 turns:  1,741
  5 turns:     91
  7 turns:    541
  9 turns:  1,979

──────────────────────────────────────────────────
PERSONA DISTRIBUTION BY DATA TYPE
──────────────────────────────────────────────────

  [CONTINUATION] (1741 total):
    moses                  587  ▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪
    jeremiah               183  ▪▪▪▪▪▪▪▪▪
    paul                   178  ▪▪▪▪▪▪▪▪▪
    david                  151  ▪▪▪▪▪▪▪
    ezekiel                149  ▪▪▪▪▪▪▪
    isai

0